<!-- source: new -->
# M5+ · Przenieś wzorzec na swoje dane

**Przebieg:** wybór danych, budowa w parach, pokaz trzech macierzy tras

Przez cały dzień budowałeś agenta dla TechRetail. Teraz ten sam wzorzec, w skrócie, na **innych danych**:

```
dane → pytania i trasy (Canvas) → funkcja UC z COMMENT → test bez modelu → (opcjonalnie) wyszukiwanie w tekście → agent → macierz tras
```

**Pracujcie w parach**: jedna osoba pisze, druga pilnuje Canvasu i macierzy tras. Dobrze, jeśli w parze jest ktoś, kto dziś utknął, i ktoś, kto skończył wcześniej.

| Opcja | Dla kogo | Dane |
|---|---|---|
| **A. Bakehouse** (domyślna) | wszyscy, zero przygotowania | `samples.bakehouse`: sprzedaż sieci piekarni i recenzje klientów, jest w każdym workspace |
| **B. Airbnb** | gdy Bakehouse jest niedostępny | `data/practice/sf_airbnb_listings.csv` (licencja MIT) |

**Karta wyjściowa (cel dnia):** macierz 3 tras Twojego agenta, z co najmniej dwiema zgodnymi, i jedno zdanie: „wzorzec, który przeniosłem, to…”.

Szablon Canvasu: `workshop/transfer/canvas_agenta.md`.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

<!-- source: new + slide 47 -->
## 1. Wybierz dane

Ustaw `DATA_OPTION` i uruchom komórkę. Powstaje tabela `workspace.default.capstone_table` i, jeśli masz tekst, `workspace.default.capstone_docs`.

Zauważ, że dla Bakehouse **nie kopiujemy** kolumny `cardNumber`. Najlepsza ochrona danych wrażliwych to nie dawać ich agentowi w ogóle.

In [ ]:
# source: new + K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py
import os
import re
import time
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

DATA_OPTION = "bakehouse"   # "bakehouse" | "airbnb"

MY_TABLE = f"{CATALOG}.{SCHEMA}.capstone_table"
MY_DOCS_TABLE = f"{CATALOG}.{SCHEMA}.capstone_docs"
DATA_DIR = Path(os.getcwd()).parent / "data"
USERNAME = spark.sql("SELECT current_user()").first()[0]

if DATA_OPTION == "bakehouse":
    spark.sql(f'''CREATE OR REPLACE TABLE {MY_TABLE} AS
        SELECT transactionID, franchiseID, dateTime, product, quantity, unitPrice, totalPrice, paymentMethod
        FROM samples.bakehouse.sales_transactions''')
    docs = spark.table("samples.bakehouse.media_customer_reviews").select(
        F.col("franchiseID").cast("string").alias("doc_id"), F.col("review").alias("content"))
else:
    listings = pd.read_csv(DATA_DIR / "practice" / "sf_airbnb_listings.csv")
    listings["price"] = pd.to_numeric(listings["price"], errors="coerce")
    listings = listings.drop(columns=["host_name", "host_id", "latitude", "longitude", "neighbourhood_group"]).dropna(subset=["id", "price"])
    listings[listings.select_dtypes("object").columns] = listings.select_dtypes("object").fillna("")
    spark.createDataFrame(listings).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MY_TABLE)
    docs = None

if docs is not None:
    (docs.where(F.length("content") > 20)
         .withColumn("chunk_id", F.sha2(F.concat_ws("||", "doc_id", "content"), 256))
         .dropDuplicates(["chunk_id"])
         .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MY_DOCS_TABLE))
HAS_DOCS = docs is not None

print(f"{MY_TABLE}: {spark.table(MY_TABLE).count():,} wierszy | tekst: {spark.table(MY_DOCS_TABLE).count() if HAS_DOCS else 'brak'}")
display(spark.table(MY_TABLE).limit(5))

<!-- source: new + slide 52 -->
## 2. Canvas w kodzie: domena, zasady i trzy trasy

Przepisz z Canvasu trzy pytania, każde z inną trasą: **funkcja**, **tekst** (albo druga funkcja) oraz **odmowa lub fallback**. Nazwy narzędzi to krótkie nazwy, które zaraz utworzysz.

In [ ]:
# source: new + slide 52
# ZADANIE C1: przepisz Canvas do kodu.
MY_DOMAIN = ...  # TODO: jedno zdanie o domenie, np. "wypożyczalnia rowerów w Krakowie"
MY_FUNCTION = f"{CATALOG}.{SCHEMA}.capstone_metric"  # możesz zmienić nazwę (małe litery, podkreślenia)
# TODO: prompt w trzech częściach: co robić, czego nie robić (dane wrażliwe), jak odmawiać / fallback
MY_SYSTEM_PROMPT = ...
# TODO: parametry testowe Twojej funkcji, np. {"requested_id": 42}
MY_TEST_PARAMETERS = ...
# TODO: trzy pytania z Canvasu; krótkie nazwy narzędzi: nazwa funkcji bez katalogu i schematu,
#       "search_my_documents" dla wyszukiwania w tekście, [] dla odmowy albo fallbacku
MY_ROUTE_CASES = [
    {"id": "t1_function", "question": ..., "expected_tools": [...]},
    {"id": "t2_text", "question": ..., "expected_tools": [...]},
    {"id": "t3_refusal", "question": ..., "expected_tools": []},
]

<!-- source: K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py + slide 28 -->
## 3. Funkcja Unity Catalog z COMMENT

Karta wzorca z M2: jedno pytanie biznesowe, `COMMENT` mówi kiedy użyć i czego nie zwraca, bez danych wrażliwych, test bez modelu.

**Podpowiedź dla Airbnb:** `requested_listing_id BIGINT` → nazwa, dzielnica, typ pokoju, cena i liczba opinii z `WHERE id = requested_listing_id`.

In [ ]:
# source: K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py + WS4[9]
# ZADANIE C2: jedna funkcja na Twojej tabeli + test bez modelu.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {MY_FUNCTION}(
  -- TODO: parametr z typem i opisem, np. requested_id BIGINT COMMENT 'Numeric ID of ...'
  TODO
)
RETURNS STRING
-- TODO: COMMENT: co zwraca, kiedy użyć, czego NIE zwraca
COMMENT 'TODO'
RETURN SELECT CONCAT(
  -- TODO: kilka kolumn Twojej tabeli jako czytelny tekst, bez danych wrażliwych
  TODO
)
FROM {MY_TABLE}
WHERE TODO
""")

from unitycatalog.ai.core.databricks import DatabricksFunctionClient

function_client = DatabricksFunctionClient(execution_mode="serverless")
print(function_client.execute_function(function_name=MY_FUNCTION, parameters=MY_TEST_PARAMETERS).value)

<!-- source: WS3[20] + slide 34 -->
## 4. (jeśli masz tekst) Wyszukiwanie w opiniach jako drugie narzędzie

Nowy indeks AI Search to dodatkowe czekanie, dlatego domyślnie narzędzie `search_my_documents` szuka po słowach kluczowych w tabeli Delta (po rdzeniu słowa, żeby łapać polską odmianę). Opinie Bakehouse są po angielsku: opis narzędzia każe agentowi przekazać angielskie słowa kluczowe. To dobry przykład, że opis narzędzia steruje nie tylko wyborem, ale też **argumentami**. To ten sam kontrakt co w M3: pytanie wchodzi, fragmenty z identyfikatorem źródła wychodzą.

**Poziom 3:** ustaw `USE_AI_SEARCH = True`. Komórka założy indeks `capstone_docs_index` na endpoincie z M3 i poczeka na jego gotowość (z limitem czasu w kodzie).

In [ ]:
# source: WS3[20] + WS4[12]
from langchain_core.tools import StructuredTool

USE_AI_SEARCH = False
MY_DOCS_INDEX = f"{CATALOG}.{SCHEMA}.capstone_docs_index"
text_tools = []

if HAS_DOCS:
    documents = spark.table(MY_DOCS_TABLE).limit(5000).toPandas()

    def search_my_documents(query: str) -> str:
        # rdzeń słowa (pierwsze 5 liter) łapie odmianę: „obsłudze” i „obsługa”, „pieczywa” i „pieczywo”
        stems = {w[:5] for w in re.findall(r"\w+", query.lower()) if len(w) > 3}
        scores = documents["content"].str.lower().apply(lambda t: sum(stem in t for stem in stems))
        top = documents.assign(score=scores).sort_values("score", ascending=False).head(4)
        return "\n\n".join(f"[{r.doc_id}] {r.content[:500]}" for r in top.itertuples() if r.score > 0) or "Brak pasujących fragmentów."

    DOCS_LANGUAGE = "English" if DATA_OPTION == "bakehouse" else "the same language as the documents"
    description = (f"Searches free-text documents about {MY_DOMAIN} (reviews, notes, descriptions) and returns excerpts with their source ID. "
                   f"Use for questions about opinions or content, not for numbers. Pass a few keywords in {DOCS_LANGUAGE}.")
    text_tools = [StructuredTool.from_function(func=search_my_documents, name="search_my_documents", description=description)]

    if USE_AI_SEARCH:
        from databricks.ai_search.client import AISearchClient
        from databricks_langchain import VectorSearchRetrieverTool

        spark.sql(f"ALTER TABLE {MY_DOCS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        client = AISearchClient(disable_notice=True)
        if not client.index_exists(SEARCH_ENDPOINT, MY_DOCS_INDEX):
            client.create_delta_sync_index(endpoint_name=SEARCH_ENDPOINT, index_name=MY_DOCS_INDEX, primary_key="chunk_id",
                                           source_table_name=MY_DOCS_TABLE, pipeline_type="TRIGGERED",
                                           embedding_source_column="content", embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
                                           columns_to_sync=["doc_id"])
        deadline = time.time() + 8 * 60
        while time.time() < deadline:
            status = client.get_index(SEARCH_ENDPOINT, MY_DOCS_INDEX).describe().get("status", {})
            if status.get("ready") or str(status.get("detailed_state", "")).upper().startswith("ONLINE"):
                text_tools = [VectorSearchRetrieverTool(index_name=MY_DOCS_INDEX, tool_name="search_my_documents",
                                                        tool_description=description, num_results=4)]
                break
            time.sleep(20)
    print(f"Narzędzie tekstowe: {'AI Search' if text_tools and not isinstance(text_tools[0], StructuredTool) else 'słowa kluczowe'}")
    print(search_my_documents(MY_ROUTE_CASES[1]["question"])[:600])
else:
    print("Brak tekstu: agent dostanie tylko funkcję. Poziom 2: dopisz drugą funkcję i dodaj ją do listy w kolejnej komórce.")

<!-- source: WS4[13] + slide 54 -->
## 5. Agent i macierz tras

Ten sam kod co w M5, tylko z Twoimi narzędziami i Twoim promptem. Tracing zapisze każdy wiersz macierzy z tagiem `route_case`. Jeśli trasa jest ❌, zrób **jedną** zmianę (COMMENT, opis narzędzia albo prompt) i uruchom komórkę jeszcze raz.

In [ ]:
# source: WS4[12] + WS4[13] + K:Warsztaty_Krzysztof/single_agent_app/notebooks/09_tagging.py
import mlflow
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

MY_EXTRA_FUNCTIONS = []  # poziom 2: pełne nazwy kolejnych funkcji UC

mlflow.set_experiment(f"/Users/{USERNAME}/{EXPERIMENT_NAME}")
mlflow.langchain.autolog()

tools = UCFunctionToolkit(function_names=[MY_FUNCTION, *MY_EXTRA_FUNCTIONS]).tools + text_tools
prompt = ChatPromptTemplate.from_messages([
    ("system", MY_SYSTEM_PROMPT), ("placeholder", "{chat_history}"), ("human", "{input}"), ("placeholder", "{agent_scratchpad}"),
])
my_agent = AgentExecutor(agent=create_tool_calling_agent(ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1), tools, prompt),
                         tools=tools, return_intermediate_steps=True, handle_parsing_errors=True, max_iterations=6)


@mlflow.trace(name="capstone_route_case")
def ask_case(case: dict) -> dict:
    mlflow.update_current_trace(tags={"route_case": case["id"], "domain": DATA_OPTION})
    return my_agent.invoke({"input": case["question"], "chat_history": []})


rows = []
for case in MY_ROUTE_CASES:
    result = ask_case(case)
    used = [step[0].tool.split("__")[-1] for step in result.get("intermediate_steps", [])]
    expected = set(case["expected_tools"])
    ok = expected.issubset(used) if expected else not used
    rows.append({"id": case["id"], "oczekiwane": ", ".join(expected) or "—", "użyte": ", ".join(used) or "—",
                 "trasa": "✅" if ok else "❌", "odpowiedź": result["output"][:220]})
    time.sleep(2)
report = pd.DataFrame(rows)
display(report)
print(f"Trasy zgodne: {(report['trasa'] == '✅').sum()}/{len(report)}  →  zrób zrzut ekranu: to Twoja karta wyjściowa")

<!-- source: new -->
## 6. Karta wyjściowa i pokaz

1. **Zrzut ekranu macierzy tras** (co najmniej 2 z 3 ✅).
2. **Jedno zdanie** w Canvasie: „wzorzec, który przeniosłem, to…” oraz „jedna rzecz, którą poprawiłem, to…”.
3. **Pokaz:** trzy pary (po jednej z każdego poziomu) pokazują macierz i jedną naprawioną trasę.

**Wyzwania, jeśli skończyłeś:**
- dodaj drugą funkcję (`MY_EXTRA_FUNCTIONS`) i czwarte pytanie, które łączy oba narzędzia;
- `USE_AI_SEARCH = True` i porównaj trasy: słowa kluczowe vs wyszukiwanie semantyczne;
- w M6 podepniesz swoją funkcję do agenta przez **MCP** bez żadnego kodu po stronie narzędzia.

<!-- source: new -->
## Podsumowanie

- Wzorzec agenta nie zależy od domeny: **pytania i trasy → narzędzia z dobrym opisem → bez danych wrażliwych → test bez modelu → agent → macierz tras**.
- Najwięcej czasu kosztuje nie kod, tylko **decyzja, jakie pytania ma obsłużyć agent i czego ma nie robić**. Dlatego zaczynamy od Canvasu.
- Narzędzie tekstowe może zacząć od wyszukiwania po słowach kluczowych, a AI Search podmieniasz, gdy kontrakt narzędzia już działa.

**Dalej:** M6. Twoja funkcja jako narzędzie MCP, ryzyka i co dodać przed PoC.